# ARIA-LLM — Fine-tune from Qwen2.5-0.5B on Google Colab (T4)

Full fp32 fine-tuning of the 358M-param model needs ~5.7 GB (weights + grads + AdamW state) plus activations — too much for a 4 GB laptop GPU, but fits on Colab's **16 GB T4** at a modest batch size.

This notebook: clone repo → rebuild the corpus with the **anti-collapse** flags → fine-tune → slim the checkpoint for deploy → (optional) upload weights to your Hugging Face model repo so the live Space can serve them.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and push your repo to GitHub first (this notebook clones it).

In [ ]:
# 1. Confirm we have a GPU (expect: Tesla T4, ~15 GB)
!nvidia-smi

Wed Jul  8 07:26:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2. Clone the repo (public). Change BRANCH if you pushed to a different one.
REPO = 'https://github.com/Aashutosh2021/ARIA_LLM.git'
BRANCH = 'main'

import os, shutil
if os.path.isdir('ARIA_LLM'):
    shutil.rmtree('ARIA_LLM')  # fresh clone so re-runs pick up new pushes
!git clone --branch $BRANCH --depth 1 $REPO
%cd ARIA_LLM

Cloning into 'ARIA_LLM'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 140 (delta 5), reused 97 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.16 MiB | 5.80 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/ARIA_LLM


In [ ]:
# 3. Install only what Colab lacks (keep Colab's CUDA torch).
!pip install -q 'transformers>=4.40' 'PyYAML>=6.0' 'tqdm>=4.65' 'huggingface_hub>=0.24'

In [ ]:
# 4. Rebuild the corpus with anti-collapse settings. --max-answer-repeats caps
#    how often any single bot answer may appear (stops one canned reply from
#    dominating and collapsing the model, e.g. the old '456 cherry drive' bug).
#    Watch the 'Most common bot answers' report it prints — no answer should be
#    a large fraction of the total. This OVERWRITES data/chat_corpus.txt.
!python scripts/build_chat_corpus.py --max-answer-repeats 3 --clean-upsample 3

Dropped 85 pairs whose answer exceeded --max-answer-repeats=3
Loaded: dialogs+csv=3639  intents=516  synthetic=3541

Most common bot answers in the corpus (after capping):
    26x  many government scholarships are supported by our university...
    25x  our university offers information technology, computer engin...
    22x  for fee detail visit <a target="_blank" href="link"> here</a...
    22x  for hostel detail visit <a target="_blank" href="add your ho...
    16x  college is open 8am-5pm monday-saturday!
    15x  college students
    15x  you can contact at: number
    14x  <a target="_blank" href="add you google map link here"> here...

Wrote 50,206 examples (5.2 MB) -> /content/ARIA_LLM/data/chat_corpus.txt

Next: train the model with:
  python train_chat.py --device cuda


In [ ]:
# 5. Download + convert Qwen weights -> checkpoints/qwen2.5-0.5b-instruct.pt
!python scripts/import_qwen.py

Downloading/Loading Qwen2.5-0.5B-Instruct via transformers...
config.json: 100% 659/659 [00:00<00:00, 3.06MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 988M/988M [00:11<00:00, 86.7MB/s]
Loading weights: 100% 290/290 [00:00<00:00, 330.67it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.05MB/s]
Converting weights...
Saving converted weights to checkpoints/qwen2.5-0.5b-instruct.pt...
Done!


In [ ]:
# 6. Fine-tune.
#  * Kaggle "T4 x2": train_qwen_finetune.py auto-detects both GPUs and uses
#    DataParallel -- it splits the batch across the two cards, so use a batch
#    that divides by 2. batch 8 => 4 samples/GPU (fits a 16GB T4 with the
#    full-attention memory cost). Drop to 4 if you OOM, or add
#    --no-data-parallel to force a single GPU.
#  * Do NOT use the P100 option: Kaggle's PyTorch dropped support for the
#    P100's older sm_60 architecture ("no kernel image available" error).
#  * Memory scales with batch_size * seq_len^2 per GPU; drop --seq-len to 128
#    if needed.
!python train_qwen_finetune.py --device cuda --epochs 5 --batch-size 8 --seq-len 256

In [ ]:
# 7. Quick sanity check across several prompts — replies should VARY (not the
#    same canned answer to everything). Higher repetition_penalty (1.3) also
#    helps avoid loops on CPU/GPU alike.
import torch
from transformers import AutoTokenizer
from chat_qwen import load_model, generate
from utils.device import get_device

device = get_device('cuda')
model, config = load_model('checkpoints_chat_qwen/best.pt', device)
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
tok.add_special_tokens({'additional_special_tokens': [config['user_token'], config['bot_token'], config['end_token']]})
stop_id = tok.convert_tokens_to_ids(config['end_token'])

for q in ['hello, who are you?', 'what can you do?', 'what is your name?', 'how are you?']:
    prompt = f"{config['user_token']} {q}\n{config['bot_token']} "
    ids = tok.encode(prompt, return_tensors='pt').to(device)
    out = [t for t in generate(model, ids, device, max_new_tokens=64, temperature=0.7,
                               top_p=0.9, repetition_penalty=1.3, max_context=512, stop_token_id=stop_id)]
    print(f'you> {q}\naria> {tok.decode(out, skip_special_tokens=True)}\n')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


you> hello, who are you?
aria> 1342.


you> what can you do?
aria> 1. i can assist with basic questions and chat about general topics.


you> what is your name?
aria>  You can call me ARIA. I am a small language model.


you> how are you?
aria> 123.




In [ ]:
# 8. Save the fine-tuned checkpoint. Either download it directly...
from google.colab import files
files.download('checkpoints_chat_qwen/best.pt')

In [ ]:
# ...or (more reliable for large files) copy it to Google Drive.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aria_checkpoints
!cp checkpoints_chat_qwen/best.pt /content/drive/MyDrive/aria_checkpoints/
print('Saved to Drive: MyDrive/aria_checkpoints/best.pt')

Mounted at /content/drive
Saved to Drive: MyDrive/aria_checkpoints/best.pt


In [ ]:
# 8. Slim both checkpoints for deployment (drops optimizer state, ~6GB -> ~1.4GB).
!python scripts/export_for_deploy.py --checkpoint checkpoints_chat_qwen/best.pt --out deploy/aria_finetuned.pt
!python scripts/export_for_deploy.py --checkpoint checkpoints/qwen2.5-0.5b-instruct.pt --out deploy/aria_qwen.pt

## 9. Upload weights to your Hugging Face model repo

Needed so the live Space (see `DEPLOY.md`) can download them. Get a **write** token at https://huggingface.co/settings/tokens . Replace `<user>` below with your HF username.

In [ ]:
from huggingface_hub import login, create_repo, upload_file

HF_USER = 'aashutosh142240'          # <-- your Hugging Face username
REPO_ID = f'{HF_USER}/aria-llm-weights'

login()  # paste your write token when prompted
create_repo(REPO_ID, repo_type='model', exist_ok=True)

for fname in ['aria_qwen.pt', 'aria_finetuned.pt']:
    upload_file(path_or_fileobj=f'deploy/{fname}', path_in_repo=fname,
                repo_id=REPO_ID, repo_type='model')
    print('uploaded', fname)
print('\nDone. Set HF_MODEL_REPO =', REPO_ID, 'in your Space settings.')